In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving fomc_sentences.csv to fomc_sentences.csv


In [ ]:
df = pd.read_csv("fomc_sentences.csv")

# Inspect the dataset
df.head()

,sentence_id,statement_id,release_date,paragraph,sentence,text,word_count,is_decision,is_admin
0,2000-02-02_statement_s01,2000-02-02_statement,2000-02-02,1,1,The Federal Open Market Committee voted today ...,23,True,False
1,2000-02-02_statement_s02,2000-02-02_statement,2000-02-02,1,2,"In a related action, the Board of Governors ap...",21,False,True
2,2000-02-02_statement_s03,2000-02-02_statement,2000-02-02,2,3,The Committee remains concerned that over time...,30,False,False
3,2000-02-02_statement_s04,2000-02-02_statement,2000-02-02,2,4,Such trends could foster inflationary imbalanc...,14,False,False
4,2000-02-02_statement_s05,2000-02-02_statement,2000-02-02,3,5,Against the background of its long-run goals o...,40,False,False


In [ ]:
print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset dimensions: (2980, 9)

Column names:
['sentence_id', 'statement_id', 'release_date', 'paragraph', 'sentence', 'text', 'word_count', 'is_decision', 'is_admin']


In [ ]:
# Check for missing values in each column
print(df.isnull().sum())

# Count administrative vs. non-administrative sentences
print(df["is_admin"].value_counts())

# Count decision vs. non-decision sentences
print(df["is_decision"].value_counts())

sentence_id     0
statement_id    0
release_date    0
paragraph       0
sentence        0
text            0
word_count      0
is_decision     0
is_admin        0
dtype: int64
is_admin
False    2859
True      121
Name: count, dtype: int64
is_decision
False    2757
True      223
Name: count, dtype: int64


In [ ]:
# Count administrative and non-administrative sentences
print(df["is_admin"].value_counts())
# 121 are admin

# Count decision and non-decision sentences
print(df["is_decision"].value_counts())
# 223 are decision

is_admin
False    2859
True      121
Name: count, dtype: int64
is_decision
False    2757
True      223
Name: count, dtype: int64


In [ ]:
# select sentences from dataset (10 random sentences)
# Display 10 randomly selected sentences from the dataset
# see language about employment, inflation, monetary policy, economic conditions
df[["release_date", "text", "is_admin", "is_decision"]].sample(10, random_state=42)

,release_date,text,is_admin,is_decision
2407,2021-09-22,The Committee seeks to achieve maximum employm...,False,False
2929,2026-01-28,In assessing the appropriate stance of monetar...,False,False
881,2011-01-26,"Although commodity prices have risen, longer-t...",False,False
331,2005-05-03,The Federal Open Market Committee decided toda...,False,True
2269,2020-07-29,"In addition, the Open Market Desk will continu...",False,False
1241,2013-09-18,"The Committee expects that, with appropriate p...",False,False
678,2009-03-18,The Committee will continue to carefully monit...,False,False
2190,2020-03-15,"Job gains have been solid, on average, in rece...",False,False
1029,2012-03-13,To support a stronger economic recovery and to...,False,False
1951,2017-12-13,The Committee expects that economic conditions...,False,False


In [ ]:
# Hugging Face Transformers
# investigate a model-based classification approach
# Count the number of unique FOMC statements
print("Number of statements:", df["statement_id"].nunique())

# Check the earliest and latest statement dates
print("Date range:", df["release_date"].min(), "to", df["release_date"].max())




Number of statements: 227
Date range: 2000-02-02 to 2026-09-16


In [ ]:
# Install Hugging Face libraries for NLP classification
!pip install -q transformers torch

In [ ]:
# Import the libraries we'll use for sentence classification
from transformers import pipeline
import torch

# Check whether Google Colab provides GPU access
print("GPU available:", torch.cuda.is_available())

GPU available: False


In [ ]:
# Load a pretrained zero-shot classification model
# pipileine give way to use pretrained model
# zero-shot classifies text into cateogires we provide
# facebook... = the pretrained model we're testing
# device =-1 --> i am running model on CPU
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
# Define the three possible tone categories
labels = ["hawkish", "dovish", "neutral"]

# Select three example sentences from our dataset
test_sentences = [
    df.iloc[0]["text"],
    df.iloc[5]["text"],
    df.iloc[10]["text"]
]

# Ask the model to classify each sentence
for sentence in test_sentences:
    result = classifier(sentence, candidate_labels=labels)

    print("\nSentence:", sentence)
    print("Predicted tone:", result["labels"][0])
    print("Confidence:", round(result["scores"][0], 3))


Sentence: The Federal Open Market Committee voted today to raise its target for the federal funds rate by 25 basis points to 5-3/4 percent.
Predicted tone: dovish
Confidence: 0.557

Sentence: In taking the discount rate action, the Federal Reserve Board approved requests submitted by the Boards of Directors of the Federal Reserve Banks of Boston, New York, Philadelphia, Cleveland, Richmond, Atlanta, Chicago, St. Louis, Kansas City and San Francisco.
Predicted tone: dovish
Confidence: 0.638

Sentence: The Committee remains concerned that increases in demand will continue to exceed the growth in potential supply, which could foster inflationary imbalances that would undermine the economy's record economic expansion.
Predicted tone: hawkish
Confidence: 0.724


Our model, facebook/bart-large-mnli, is a general-purpose zero-shot classifier.

It understands language, but it hasn't specifically learned how central banks communicate monetary policy.

This is why we shouldn't immediately classify all 2,980 sentences using this model.

In [ ]:
# Load a pretrained central-bank tone classification model
from transformers import pipeline

fomc_classifier = pipeline(
    "text-classification",
    model="Woochachino/cbsent",
    device=-1
)

config.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

In [ ]:
# Test the new model on the same sentences as before
for sentence in test_sentences:
    result = fomc_classifier(sentence)

    print("\nSentence:", sentence)
    print("Predicted tone:", result[0]["label"])
    print("Confidence:", round(result[0]["score"], 3))


Sentence: The Federal Open Market Committee voted today to raise its target for the federal funds rate by 25 basis points to 5-3/4 percent.
Predicted tone: hawkish
Confidence: 0.987

Sentence: In taking the discount rate action, the Federal Reserve Board approved requests submitted by the Boards of Directors of the Federal Reserve Banks of Boston, New York, Philadelphia, Cleveland, Richmond, Atlanta, Chicago, St. Louis, Kansas City and San Francisco.
Predicted tone: neutral
Confidence: 1.0

Sentence: The Committee remains concerned that increases in demand will continue to exceed the growth in potential supply, which could foster inflationary imbalances that would undermine the economy's record economic expansion.
Predicted tone: hawkish
Confidence: 1.0


YES! This is a much better result. Our new model correctly classified all three examples.

Unlike BART, CBSent was developed for central-bank communication. It recognizes monetary-policy language rather than relying on general language associations.

One important distinction: these three examples are a preliminary sanity check, not proof of accuracy. We still need to evaluate it on a larger sample.

In [ ]:
# Select a sample from different sentence categories
admin_sample = df[df["is_admin"] == True].sample(5, random_state=42)

decision_sample = df[
    (df["is_decision"] == True) & (df["is_admin"] == False)
].sample(5, random_state=42)

regular_sample = df[
    (df["is_admin"] == False) & (df["is_decision"] == False)
].sample(10, random_state=42)

# Combine the samples into one testing dataset
test_df = pd.concat([
    admin_sample,
    decision_sample,
    regular_sample
])

# Ask CBSent to classify each sentence
test_df["prediction"] = test_df["text"].apply(
    lambda x: fomc_classifier(x)[0]["label"]
)

# Display the results for manual inspection
test_df[[
    "release_date",
    "text",
    "is_admin",
    "is_decision",
    "prediction"
]]

,release_date,text,is_admin,is_decision,prediction
290,2004-09-21,"In taking this action, the Board approved the ...",True,False,neutral
307,2004-12-14,"In a related action, the Board of Governors un...",True,False,dovish
12,2000-03-21,"In taking the discount rate action, the Federa...",True,False,neutral
330,2005-03-22,"In taking this action, the Board approved the ...",True,False,neutral
119,2001-10-02,"In a related action, the Board of Governors ap...",True,False,neutral
56,2001-01-31,The Federal Open Market Committee at its meeti...,False,True,dovish
754,2009-12-16,The Committee will maintain the target range f...,False,True,dovish
1273,2013-10-30,To support continued progress toward maximum e...,False,True,dovish
1814,2017-02-01,In view of realized and expected labor market ...,False,True,neutral
2957,2026-06-17,The Committee decided to maintain the target r...,False,True,neutral


Let's see how confident CBSent was in these 20 predictions. This will help us identify sentences that may need closer review before classifying the entire dataset.

In [ ]:
# Get each prediction and its confidence score
results = fomc_classifier(test_df["text"].tolist())

test_df["prediction"] = [result["label"] for result in results]
test_df["confidence"] = [result["score"] for result in results]

# Display the predictions and confidence scores
test_df[[
    "release_date",
    "text",
    "is_admin",
    "is_decision",
    "prediction",
    "confidence"
]]

,release_date,text,is_admin,is_decision,prediction,confidence
290,2004-09-21,"In taking this action, the Board approved the ...",True,False,neutral,0.999350
307,2004-12-14,"In a related action, the Board of Governors un...",True,False,dovish,0.917113
12,2000-03-21,"In taking the discount rate action, the Federa...",True,False,neutral,0.999961
330,2005-03-22,"In taking this action, the Board approved the ...",True,False,neutral,0.999653
119,2001-10-02,"In a related action, the Board of Governors ap...",True,False,neutral,0.533890
56,2001-01-31,The Federal Open Market Committee at its meeti...,False,True,dovish,0.999977
754,2009-12-16,The Committee will maintain the target range f...,False,True,dovish,0.999998
1273,2013-10-30,To support continued progress toward maximum e...,False,True,dovish,0.973365
1814,2017-02-01,In view of realized and expected labor market ...,False,True,neutral,0.999999
2957,2026-06-17,The Committee decided to maintain the target r...,False,True,neutral,0.999819


What stands out
- 18 of 20 predictions have confidence above 0.80.
- Two sentences have noticeably lower confidence:
  - Row 119: neutral, 0.534
  - Row 52: neutral, 0.478

  The administrative sentence classified as dovish (row 307) has high confidence, but it won't affect your statement scores because administrative sentences are excluded.

In [ ]:
# Next step: Inspect the two uncertain sentences

# Display sentences where the model has low confidence
low_confidence = test_df[test_df["confidence"] < 0.80]

pd.set_option("display.max_colwidth", None)

display(low_confidence[[
    "release_date",
    "text",
    "is_admin",
    "is_decision",
    "prediction",
    "confidence"
]])

,release_date,text,is_admin,is_decision,prediction,confidence
119,2001-10-02,"In a related action, the Board of Governors approved a 50 basis point reduction in the discount rate to 2 percent.",True,False,neutral,0.533890
52,2001-01-03,"Moreover, inflation pressures remain contained.",False,False,neutral,0.478001


My recommendation: Keep CBSent as our preliminary classifier. These examples don't reveal a clear reason to reject it, but we'll still treat its predictions as model-generated estimates, not verified labels.

In [ ]:
# Classify all 2,980 Sentences
# Now we'll apply the model to the entire dataset and save both the predicted tone and
# confidence score for every sentence.

# Classify all sentences using CBSent
all_results = fomc_classifier(
    df["text"].tolist(),
    batch_size=16
)

# Save the predicted tone and confidence
df["predicted_tone"] = [result["label"].lower() for result in all_results]
df["confidence"] = [result["score"] for result in all_results]

# Check the classification results
print(df["predicted_tone"].value_counts())
print("\nAverage confidence:", df["confidence"].mean())

predicted_tone
neutral    1324
dovish     1126
hawkish     530
Name: count, dtype: int64

Average confidence: 0.938016650750733


predicted_tone
neutral    1324 (44%)
dovish     1126 (38%)
hawkish     530 (18%)
Name: count, dtype: int64

Average confidence: 0.938016650750733

The model classified more sentences as dovish than hawkish. That's an interesting initial result, but it does not necessarily mean the Fed was dovish overall. We still need to calculate the statement-level scores.

In [ ]:
# Calculate statement-level tone scores
def calculate_scores(data):
    return (
        data.groupby(["statement_id", "release_date"])
        .agg(
            total_sentences=("predicted_tone", "size"),
            hawkish=("predicted_tone", lambda x: (x == "hawkish").sum()),
            dovish=("predicted_tone", lambda x: (x == "dovish").sum())
        )
        .assign(
            tone_score=lambda x: (
                (x["hawkish"] - x["dovish"]) / x["total_sentences"] * 100
            )
        )
        .reset_index()
    )

# Version 1: Exclude admin sentences, include decision sentences
with_decision = calculate_scores(
    df[df["is_admin"] == False]
)

# Version 2: Exclude admin and decision sentences
without_decision = calculate_scores(
    df[(df["is_admin"] == False) & (df["is_decision"] == False)]
)

# Combine both scores into one table
statement_scores = with_decision.merge(
    without_decision,
    on=["statement_id", "release_date"],
    suffixes=("_with_decision", "_without_decision")
)

# Sort chronologically and calculate changes from the previous statement
statement_scores["release_date"] = pd.to_datetime(
    statement_scores["release_date"]
)

statement_scores = statement_scores.sort_values("release_date")

statement_scores["tone_shift_with_decision"] = (
    statement_scores["tone_score_with_decision"].diff()
)

statement_scores["tone_shift_without_decision"] = (
    statement_scores["tone_score_without_decision"].diff()
)

# Display the results
display(statement_scores.head(10))

,statement_id,release_date,total_sentences_with_decision,hawkish_with_decision,dovish_with_decision,tone_score_with_decision,total_sentences_without_decision,hawkish_without_decision,dovish_without_decision,tone_score_without_decision,tone_shift_with_decision,tone_shift_without_decision
0,2000-02-02_statement,2000-02-02,4,4,0,100.000000,3,3,0,100.000000,NaN,NaN
1,2000-03-21_statement,2000-03-21,4,3,0,75.000000,3,2,0,66.666667,-25.000000,-33.333333
2,2000-05-16_statement,2000-05-16,4,4,0,100.000000,3,3,0,100.000000,25.000000,33.333333
3,2000-06-28_statement,2000-06-28,5,4,0,80.000000,4,4,0,100.000000,-20.000000,0.000000
4,2000-08-22_statement,2000-08-22,5,4,0,80.000000,4,4,0,100.000000,0.000000,0.000000
5,2000-10-03_statement,2000-10-03,7,5,0,71.428571,6,5,0,83.333333,-8.571429,-16.666667
6,2000-11-15_statement,2000-11-15,5,4,0,80.000000,4,4,0,100.000000,8.571429,16.666667
7,2000-12-19_statement,2000-12-19,5,0,2,-40.000000,4,0,2,-50.000000,-120.000000,-150.000000
8,2001-01-03_statement,2001-01-03,5,1,2,-20.000000,4,1,1,0.000000,20.000000,50.000000
9,2001-01-31_statement,2001-01-31,7,3,2,14.285714,6,3,1,33.333333,34.285714,33.333333


In [ ]:
# Verify we retained all 227 statements and didn't lose any during merge
# Check the number of statements and missing values
print("Original statements:", df["statement_id"].nunique())
print("Statements with decision:", len(with_decision))
print("Statements without decision:", len(without_decision))
print("Statements in final table:", len(statement_scores))

print("\nMissing values:")
print(statement_scores.isnull().sum())

print("\nDuplicate statement IDs:")
print(statement_scores["statement_id"].duplicated().sum())

Original statements: 227
Statements with decision: 227
Statements without decision: 227
Statements in final table: 227

Missing values:
statement_id                        0
release_date                        0
total_sentences_with_decision       0
hawkish_with_decision               0
dovish_with_decision                0
tone_score_with_decision            0
total_sentences_without_decision    0
hawkish_without_decision            0
dovish_without_decision             0
tone_score_without_decision         0
tone_shift_with_decision            1
tone_shift_without_decision         1
dtype: int64

Duplicate statement IDs:
0


In [ ]:
# Prepare the sentence-level dataset
sentence_scores = df[[
    "sentence_id",
    "statement_id",
    "release_date",
    "text",
    "is_admin",
    "is_decision",
    "predicted_tone",
    "confidence"
]].copy()

# Export both datasets
sentence_scores.to_csv("fomc_sentence_scores.csv", index=False)
statement_scores.to_csv("fomc_statement_scores.csv", index=False)

# Download both files
from google.colab import files

files.download("fomc_sentence_scores.csv")
files.download("fomc_statement_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FEW SHOT CLASSIFICATION
- 10 dovish sentences
- 10 neutral sentences
- give to AI model and ask it to classify the remaining FOMC sentences based on the patterns

1. Pretrained Model
- used CBSent
- already trained on central-bank knowledge
- dont know how accururately it classifies your particular FOMC data until you validate it

2. Give AI 10 examples
- few-shot prompting
- give model examples, ask it to classify remaining sentences
- define exactly what hawkish and dovish mean for the project
- model gets examples of the classifications your team expects
- don't need to train new model
- not enough examples
- giving examples to model doesn't permanently train it


3. Fine-tune
- more advanced
- take existing model and train further using sentences your team has manually labeled
- model adjusts parameters to learn team's classification standard

Model 1 -- CBSent (pretrained model(


Model 2 -- Few-shot classification
- 10 example sentneces

Model 3 -- Ensemble classifier

In [ ]:
%pip install -q setfit datasets

In [ ]:
# Select 5 candidate sentences from each CBSent category
candidates = []

for label in ["hawkish", "dovish", "neutral"]:
    sample = df[
        (df["predicted_tone"] == label) &
        (df["is_admin"] == False)
    ].sample(5, random_state=42)

    candidates.append(sample)

training_candidates = pd.concat(candidates)
training_candidates = training_candidates.sample(
    frac=1, random_state=42
).reset_index(drop=True)

# Create a blank column for human-assigned labels
training_candidates["human_label"] = ""

display(training_candidates[[
    "sentence_id",
    "text",
    "predicted_tone",
    "human_label"
]])

,sentence_id,text,predicted_tone,human_label
0,2013-06-19_statement_s02,"Labor market conditions have shown further improvement in recent months, on balance, but the unemployment rate remains elevated.",dovish,
1,2004-12-14_statement_s07,"Nonetheless, the Committee will respond to changes in economic prospects as needed to fulfill its obligation to maintain price stability.",neutral,
2,2007-06-28_statement_s06,"Moreover, the high level of resource utilization has the potential to sustain those pressures.",hawkish,
3,2003-09-16_statement_s04,Business pricing power and increases in core consumer prices remain muted.,neutral,
4,2023-02-01_statement_s06,The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.,dovish,
5,2012-09-13_statement_s15,"If the outlook for the labor market does not improve substantially, the Committee will continue its purchases of agency mortgage-backed securities, undertake additional asset purchases, and employ its other policy tools as appropriate until such improvement is achieved in a context of price stability.",dovish,
6,2000-03-21_statement_s05,"Against the background of its long-run goals of price stability and sustainable economic growth and of the information currently available, the Committee believes the risks are weighted mainly toward conditions that may generate heightened inflation pressures in the foreseeable future.",hawkish,
7,2022-03-16_statement_s08,"In support of these goals, the Committee decided to raise the target range for the federal funds rate to 1/4 to 1/2 percent and anticipates that ongoing increases in the target range will be appropriate.",hawkish,
8,2023-09-20_statement_s14,"In assessing the appropriate stance of monetary policy, the Committee will continue to monitor the implications of incoming information for the economic outlook.",neutral,
9,2018-01-31_statement_s01,Information received since the Federal Open Market Committee met in December indicates that the labor market has continued to strengthen and that economic activity has been rising at a solid rate.,hawkish,


In [ ]:
# Assign our proposed labels to the 15 examples
# HAVE TEAM REVIEW before treating them as ground truth
labels = {
    "2013-06-19_statement_s02": "dovish",
    "2004-12-14_statement_s07": "neutral",
    "2007-06-28_statement_s06": "hawkish",
    "2003-09-16_statement_s04": "dovish",
    "2023-02-01_statement_s06": "neutral",
    "2012-09-13_statement_s15": "dovish",
    "2000-03-21_statement_s05": "hawkish",
    "2022-03-16_statement_s08": "hawkish",
    "2023-09-20_statement_s14": "neutral",
    "2018-01-31_statement_s01": "hawkish",
    "2025-12-10_statement_s08": "dovish",
    "2014-06-18_statement_s20": "neutral",
    "2003-05-06_statement_s05": "neutral",
    "2018-08-01_statement_s02": "hawkish",
    "2016-11-02_statement_s17": "hawkish"
}

training_candidates["human_label"] = (
    training_candidates["sentence_id"].map(labels)
)

display(training_candidates[[
    "sentence_id", "text", "human_label"
]])

,sentence_id,text,human_label
0,2013-06-19_statement_s02,"Labor market conditions have shown further improvement in recent months, on balance, but the unemployment rate remains elevated.",dovish
1,2004-12-14_statement_s07,"Nonetheless, the Committee will respond to changes in economic prospects as needed to fulfill its obligation to maintain price stability.",neutral
2,2007-06-28_statement_s06,"Moreover, the high level of resource utilization has the potential to sustain those pressures.",hawkish
3,2003-09-16_statement_s04,Business pricing power and increases in core consumer prices remain muted.,dovish
4,2023-02-01_statement_s06,The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.,neutral
5,2012-09-13_statement_s15,"If the outlook for the labor market does not improve substantially, the Committee will continue its purchases of agency mortgage-backed securities, undertake additional asset purchases, and employ its other policy tools as appropriate until such improvement is achieved in a context of price stability.",dovish
6,2000-03-21_statement_s05,"Against the background of its long-run goals of price stability and sustainable economic growth and of the information currently available, the Committee believes the risks are weighted mainly toward conditions that may generate heightened inflation pressures in the foreseeable future.",hawkish
7,2022-03-16_statement_s08,"In support of these goals, the Committee decided to raise the target range for the federal funds rate to 1/4 to 1/2 percent and anticipates that ongoing increases in the target range will be appropriate.",hawkish
8,2023-09-20_statement_s14,"In assessing the appropriate stance of monetary policy, the Committee will continue to monitor the implications of incoming information for the economic outlook.",neutral
9,2018-01-31_statement_s01,Information received since the Federal Open Market Committee met in December indicates that the labor market has continued to strengthen and that economic activity has been rising at a solid rate.,hawkish


In [ ]:
# Check that every sentence has a human label
print("Missing labels:", training_candidates["human_label"].isna().sum())

# Count examples in each category
print("\nLabel distribution:")
print(training_candidates["human_label"].value_counts())

# Check that SetFit is installed
import setfit
print("\nSetFit installed successfully!")

Missing labels: 0

Label distribution:
human_label
hawkish    6
neutral    5
dovish     4
Name: count, dtype: int64

SetFit installed successfully!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from datasets import Dataset

# Keep only the sentence and its human-assigned label
train_data = training_candidates[[
    "text",
    "human_label"
]].copy()

# Rename columns for SetFit
train_data = train_data.rename(columns={
    "text": "text",
    "human_label": "label"
})

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(
    train_data,
    preserve_index=False
)

print(train_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 15
})


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from setfit import SetFitModel

# Load a lightweight pretrained sentence model
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=["hawkish", "dovish", "neutral"]
)

print("SetFit model loaded successfully!")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (8 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


SetFit model loaded successfully!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# TRAIN MODEL 2
from setfit import Trainer, TrainingArguments

# Set training parameters
args = TrainingArguments(
    batch_size=4,
    num_epochs=3
)

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset
)

# Train the model
trainer.train()

print("Model 2 training complete!")

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 148
  Batch size = 4
  Num epochs = 3
/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.363771
50,0.214274
100,0.122420


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Model 2 training complete!


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Select 10 sentences that were NOT used for training
test_sample = df[
    ~df["sentence_id"].isin(training_candidates["sentence_id"])
].sample(10, random_state=42).copy()

# Get Model 2 predictions
test_sample["setfit_prediction"] = list(
    model.predict(test_sample["text"].tolist())
)

# Compare both models
display(test_sample[[
    "text",
    "predicted_tone",
    "setfit_prediction"
]])

,text,predicted_tone,setfit_prediction
2896,The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.,dovish,neutral
2082,"In support of these goals, the Committee decided to maintain the target range for the federal funds rate at 2-1/4 to 2-1/2 percent.",neutral,neutral
2739,The Committee will maintain the monthly redemption cap on agency debt and agency mortgage-backed securities at $35 billion and will reinvest any principal payments in excess of this cap into Treasury securities.,dovish,neutral
1692,"Market-based measures of inflation compensation remain low; survey-based measures of longer-term inflation expectations are little changed, on balance, in recent months.",neutral,dovish
2705,"In support of its goals, the Committee decided to maintain the target range for the federal funds rate at 5-1/4 to 5-1/2 percent.",neutral,neutral
1427,"Consistent with its statutory mandate, the Committee seeks to foster maximum employment and price stability.",dovish,neutral
1330,The Committee sees the risks to the outlook for the economy and the labor market as nearly balanced.,neutral,neutral
815,Housing starts remain at a depressed level.,dovish,hawkish
1680,"In determining the timing and size of future adjustments to the target range for the federal funds rate, the Committee will assess realized and expected economic conditions relative to its objectives of maximum employment and 2 percent inflation.",neutral,neutral
336,Longer-term inflation expectations remain well contained.,neutral,hawkish


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Identify agreement between the two models
test_sample["agreement"] = (
    test_sample["predicted_tone"] ==
    test_sample["setfit_prediction"]
)

# Summarize agreement
print("Agreement summary:")
print(test_sample["agreement"].value_counts())

# Show only disagreements
disagreements = test_sample[
    test_sample["agreement"] == False
]

display(disagreements[[
    "text",
    "predicted_tone",
    "setfit_prediction"
]])

Agreement summary:
agreement
False    6
True     4
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,text,predicted_tone,setfit_prediction
2896,The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.,dovish,neutral
2739,The Committee will maintain the monthly redemption cap on agency debt and agency mortgage-backed securities at $35 billion and will reinvest any principal payments in excess of this cap into Treasury securities.,dovish,neutral
1692,"Market-based measures of inflation compensation remain low; survey-based measures of longer-term inflation expectations are little changed, on balance, in recent months.",neutral,dovish
1427,"Consistent with its statutory mandate, the Committee seeks to foster maximum employment and price stability.",dovish,neutral
815,Housing starts remain at a depressed level.,dovish,hawkish
336,Longer-term inflation expectations remain well contained.,neutral,hawkish


In [ ]:
# Create an ensemble prediction
test_sample["ensemble_prediction"] = test_sample.apply(
    lambda row: row["predicted_tone"]
    if row["predicted_tone"] == row["setfit_prediction"]
    else "needs_review",
    axis=1
)

# Display the combined results
display(test_sample[[
    "text",
    "predicted_tone",
    "setfit_prediction",
    "ensemble_prediction"
]])

,text,predicted_tone,setfit_prediction,ensemble_prediction
2896,The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run.,dovish,neutral,needs_review
2082,"In support of these goals, the Committee decided to maintain the target range for the federal funds rate at 2-1/4 to 2-1/2 percent.",neutral,neutral,neutral
2739,The Committee will maintain the monthly redemption cap on agency debt and agency mortgage-backed securities at $35 billion and will reinvest any principal payments in excess of this cap into Treasury securities.,dovish,neutral,needs_review
1692,"Market-based measures of inflation compensation remain low; survey-based measures of longer-term inflation expectations are little changed, on balance, in recent months.",neutral,dovish,needs_review
2705,"In support of its goals, the Committee decided to maintain the target range for the federal funds rate at 5-1/4 to 5-1/2 percent.",neutral,neutral,neutral
1427,"Consistent with its statutory mandate, the Committee seeks to foster maximum employment and price stability.",dovish,neutral,needs_review
1330,The Committee sees the risks to the outlook for the economy and the labor market as nearly balanced.,neutral,neutral,neutral
815,Housing starts remain at a depressed level.,dovish,hawkish,needs_review
1680,"In determining the timing and size of future adjustments to the target range for the federal funds rate, the Committee will assess realized and expected economic conditions relative to its objectives of maximum employment and 2 percent inflation.",neutral,neutral,neutral
336,Longer-term inflation expectations remain well contained.,neutral,hawkish,needs_review


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Generate SetFit predictions for the full dataset
df["setfit_prediction"] = list(
    model.predict(df["text"].tolist())
)

# Combine the two models
df["ensemble_prediction"] = df.apply(
    lambda row: row["predicted_tone"]
    if row["predicted_tone"] == row["setfit_prediction"]
    else "needs_review",
    axis=1
)

# Count the final classifications
print("Ensemble classification counts:")
print(df["ensemble_prediction"].value_counts())

print("\nTotal sentences:", len(df))

Ensemble classification counts:
ensemble_prediction
needs_review    1807
neutral          657
hawkish          365
dovish           151
Name: count, dtype: int64

Total sentences: 2980


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Exclude administrative sentences
analysis_df = df[df["is_admin"] == False].copy()

# Calculate agreement percentages
agreement_rate = (
    analysis_df["ensemble_prediction"] != "needs_review"
).mean() * 100

review_rate = (
    analysis_df["ensemble_prediction"] == "needs_review"
).mean() * 100

print("Non-administrative sentences:", len(analysis_df))
print(f"Model agreement: {agreement_rate:.2f}%")
print(f"Human review needed: {review_rate:.2f}%")

Non-administrative sentences: 2859
Model agreement: 38.44%
Human review needed: 61.56%


In [ ]:
# Compare the individual model predictions
print("CBSent distribution:")
print(df["predicted_tone"].value_counts(normalize=True).mul(100).round(2))

print("\nSetFit distribution:")
print(df["setfit_prediction"].value_counts(normalize=True).mul(100).round(2))

CBSent distribution:
predicted_tone
neutral    44.43
dovish     37.79
hawkish    17.79
Name: proportion, dtype: float64

SetFit distribution:
setfit_prediction
hawkish    48.99
neutral    42.58
dovish      8.42
Name: proportion, dtype: float64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Select 30 non-administrative sentences
# Exclude the 15 training examples

validation_candidates = df[
    (df["is_admin"] == False) &
    (~df["sentence_id"].isin(training_candidates["sentence_id"]))
].copy()

# Sample 10 sentences from each CBSent category
validation_sample = (
    validation_candidates
    .groupby("predicted_tone", group_keys=False)
    .sample(n=10, random_state=123)
    .reset_index(drop=True)
)

# Add blank column for independent human labels
validation_sample["human_label"] = ""

display(validation_sample[[
    "sentence_id",
    "text",
    "predicted_tone",
    "setfit_prediction",
    "human_label"
]])

,sentence_id,text,predicted_tone,setfit_prediction,human_label
0,2014-01-29_statement_s13,"Beginning in February, the Committee will add to its holdings of agency mortgage-backed securities at a pace of $30 billion per month rather than $35 billion per month, and will add to its holdings of longer-term Treasury securities at a pace of $35 billion per month rather than $40 billion per month.",dovish,hawkish,
1,2006-09-20_statement_s04,"However, inflation pressures seem likely to moderate over time, reflecting reduced impetus from energy prices, contained inflation expectations, and the cumulative effects of monetary policy actions and other factors restraining aggregate demand.",dovish,hawkish,
2,2009-08-12_statement_s07,"However, substantial resource slack is likely to dampen cost pressures, and the Committee expects that inflation will remain subdued for some time.",dovish,hawkish,
3,2010-09-21_statement_s06,"Bank lending has continued to contract, but at a reduced rate in recent months.",dovish,dovish,
4,2013-12-18_statement_s21,"The Committee now anticipates, based on its assessment of these factors, that it likely will be appropriate to maintain the current target range for the federal funds rate well past the time that the unemployment rate declines below 6-1/2 percent, especially if projected inflation continues to run below the Committee's 2 percent longer-run goal.",dovish,hawkish,
5,2001-04-18_statement_s08,"This potential restraint, together with the possible effects of earlier reductions in equity wealth on consumption and the risk of slower growth abroad, threatens to keep the pace of economic activity unacceptably weak.",dovish,hawkish,
6,2024-03-20_statement_s09,The Committee does not expect it will be appropriate to reduce the target range until it has gained greater confidence that inflation is moving sustainably toward 2 percent.,dovish,neutral,
7,2014-09-17_statement_s09,The Committee sees the risks to the outlook for economic activity and the labor market as nearly balanced and judges that the likelihood of inflation running persistently below 2 percent has diminished somewhat since early this year.,dovish,neutral,
8,2023-03-22_statement_s11,The Committee anticipates that some additional policy firming may be appropriate in order to attain a stance of monetary policy that is sufficiently restrictive to return inflation to 2 percent over time.,dovish,neutral,
9,2012-06-20_statement_s02,"However, growth in employment has slowed in recent months, and the unemployment rate remains elevated.",dovish,hawkish,


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
proposed_labels = [
    "hawkish", "dovish", "dovish", "dovish", "dovish",
    "dovish", "hawkish", "hawkish", "hawkish", "dovish",
    "hawkish", "hawkish", "hawkish", "dovish", "neutral",
    "hawkish", "dovish", "hawkish", "hawkish", "neutral",
    "dovish", "neutral", "neutral", "neutral", "neutral",
    "neutral", "neutral", "neutral", "dovish", "neutral"
]

validation_sample["proposed_label"] = proposed_labels

display(validation_sample[[
    "sentence_id",
    "text",
    "predicted_tone",
    "setfit_prediction",
    "proposed_label"
]])

,sentence_id,text,predicted_tone,setfit_prediction,proposed_label
0,2014-01-29_statement_s13,"Beginning in February, the Committee will add to its holdings of agency mortgage-backed securities at a pace of $30 billion per month rather than $35 billion per month, and will add to its holdings of longer-term Treasury securities at a pace of $35 billion per month rather than $40 billion per month.",dovish,hawkish,hawkish
1,2006-09-20_statement_s04,"However, inflation pressures seem likely to moderate over time, reflecting reduced impetus from energy prices, contained inflation expectations, and the cumulative effects of monetary policy actions and other factors restraining aggregate demand.",dovish,hawkish,dovish
2,2009-08-12_statement_s07,"However, substantial resource slack is likely to dampen cost pressures, and the Committee expects that inflation will remain subdued for some time.",dovish,hawkish,dovish
3,2010-09-21_statement_s06,"Bank lending has continued to contract, but at a reduced rate in recent months.",dovish,dovish,dovish
4,2013-12-18_statement_s21,"The Committee now anticipates, based on its assessment of these factors, that it likely will be appropriate to maintain the current target range for the federal funds rate well past the time that the unemployment rate declines below 6-1/2 percent, especially if projected inflation continues to run below the Committee's 2 percent longer-run goal.",dovish,hawkish,dovish
5,2001-04-18_statement_s08,"This potential restraint, together with the possible effects of earlier reductions in equity wealth on consumption and the risk of slower growth abroad, threatens to keep the pace of economic activity unacceptably weak.",dovish,hawkish,dovish
6,2024-03-20_statement_s09,The Committee does not expect it will be appropriate to reduce the target range until it has gained greater confidence that inflation is moving sustainably toward 2 percent.,dovish,neutral,hawkish
7,2014-09-17_statement_s09,The Committee sees the risks to the outlook for economic activity and the labor market as nearly balanced and judges that the likelihood of inflation running persistently below 2 percent has diminished somewhat since early this year.,dovish,neutral,hawkish
8,2023-03-22_statement_s11,The Committee anticipates that some additional policy firming may be appropriate in order to attain a stance of monetary policy that is sufficiently restrictive to return inflation to 2 percent over time.,dovish,neutral,hawkish
9,2012-06-20_statement_s02,"However, growth in employment has slowed in recent months, and the unemployment rate remains elevated.",dovish,hawkish,dovish


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Use our proposed labels as temporary reference labels
y_true = validation_sample["proposed_label"]

# Model 1: CBSent
y_cbsent = validation_sample["predicted_tone"]

# Model 2: SetFit
y_setfit = validation_sample["setfit_prediction"]

print("MODEL 1: CBSent")
print("Accuracy:", accuracy_score(y_true, y_cbsent))
print(classification_report(y_true, y_cbsent, zero_division=0))

print("\nMODEL 2: SetFit")
print("Accuracy:", accuracy_score(y_true, y_setfit))
print(classification_report(y_true, y_setfit, zero_division=0))

MODEL 1: CBSent
Accuracy: 0.6666666666666666
              precision    recall  f1-score   support

      dovish       0.60      0.60      0.60        10
     hawkish       0.60      0.60      0.60        10
     neutral       0.80      0.80      0.80        10

    accuracy                           0.67        30
   macro avg       0.67      0.67      0.67        30
weighted avg       0.67      0.67      0.67        30


MODEL 2: SetFit
Accuracy: 0.3333333333333333
              precision    recall  f1-score   support

      dovish       1.00      0.10      0.18        10
     hawkish       0.22      0.40      0.29        10
     neutral       0.45      0.50      0.48        10

    accuracy                           0.33        30
   macro avg       0.56      0.33      0.31        30
weighted avg       0.56      0.33      0.31        30



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Important research limitation
I need to emphasize something: these are not independent validation results. I proposed the 30 reference labels after seeing both models' predictions. That introduces potential bias.

We should not report 66.7% as CBSent's validated accuracy in your research paper.

In [ ]:
# My final AI-assisted annotations
final_labels = [
    "hawkish", "dovish", "dovish", "neutral", "dovish",
    "neutral", "hawkish", "hawkish", "hawkish", "neutral",
    "hawkish", "hawkish", "hawkish", "neutral", "neutral",
    "hawkish", "dovish", "hawkish", "hawkish", "neutral",
    "neutral", "neutral", "neutral", "neutral", "neutral",
    "neutral", "neutral", "neutral", "dovish", "neutral"
]

validation_sample["reference_label"] = final_labels

display(validation_sample[[
    "sentence_id",
    "predicted_tone",
    "setfit_prediction",
    "reference_label"
]])

,sentence_id,predicted_tone,setfit_prediction,reference_label
0,2014-01-29_statement_s13,dovish,hawkish,hawkish
1,2006-09-20_statement_s04,dovish,hawkish,dovish
2,2009-08-12_statement_s07,dovish,hawkish,dovish
3,2010-09-21_statement_s06,dovish,dovish,neutral
4,2013-12-18_statement_s21,dovish,hawkish,dovish
5,2001-04-18_statement_s08,dovish,hawkish,neutral
6,2024-03-20_statement_s09,dovish,neutral,hawkish
7,2014-09-17_statement_s09,dovish,neutral,hawkish
8,2023-03-22_statement_s11,dovish,neutral,hawkish
9,2012-06-20_statement_s02,dovish,hawkish,neutral


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true = validation_sample["reference_label"]

print("CBSent:")
print("Accuracy:", accuracy_score(
    y_true, validation_sample["predicted_tone"]
))
print(classification_report(
    y_true, validation_sample["predicted_tone"],
    zero_division=0
))

print("\nSetFit:")
print("Accuracy:", accuracy_score(
    y_true, validation_sample["setfit_prediction"]
))
print(classification_report(
    y_true, validation_sample["setfit_prediction"],
    zero_division=0
))

CBSent:
Accuracy: 0.6
              precision    recall  f1-score   support

      dovish       0.30      0.60      0.40         5
     hawkish       0.60      0.60      0.60        10
     neutral       0.90      0.60      0.72        15

    accuracy                           0.60        30
   macro avg       0.60      0.60      0.57        30
weighted avg       0.70      0.60      0.63        30


SetFit:
Accuracy: 0.3
              precision    recall  f1-score   support

      dovish       0.00      0.00      0.00         5
     hawkish       0.22      0.40      0.29        10
     neutral       0.45      0.33      0.38        15

    accuracy                           0.30        30
   macro avg       0.23      0.24      0.22        30
weighted avg       0.30      0.30      0.29        30



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

# 1. Create sentence-level dataset using CBSent

sentence_scores = df[[
    "sentence_id",
    "statement_id",
    "release_date",
    "text",
    "is_admin",
    "is_decision",
    "predicted_tone",
    "confidence"
]].copy()

# 2. Calculate statement-level tone scores

def calculate_scores(data):
    return (
        data.groupby(["statement_id", "release_date"])
        .agg(
            total_sentences=("predicted_tone", "size"),
            hawkish=("predicted_tone", lambda x: (x == "hawkish").sum()),
            dovish=("predicted_tone", lambda x: (x == "dovish").sum()),
            neutral=("predicted_tone", lambda x: (x == "neutral").sum())
        )
        .assign(
            tone_score=lambda x:
                (x["hawkish"] - x["dovish"])
                / x["total_sentences"] * 100
        )
        .reset_index()
    )

# Include decision sentences, exclude administrative sentences
with_decision = calculate_scores(
    df[df["is_admin"] == False]
)

# Exclude both administrative and decision sentences
without_decision = calculate_scores(
    df[
        (df["is_admin"] == False) &
        (df["is_decision"] == False)
    ]
)

# 3. Combine both sets of statement scores
statement_scores = with_decision.merge(
    without_decision,
    on=["statement_id", "release_date"],
    suffixes=("_with_decision", "_without_decision")
)

# 4. Calculate changes from previous statements
statement_scores["release_date"] = pd.to_datetime(
    statement_scores["release_date"]
)

statement_scores = statement_scores.sort_values("release_date")

statement_scores["tone_shift_with_decision"] = (
    statement_scores["tone_score_with_decision"].diff()
)

statement_scores["tone_shift_without_decision"] = (
    statement_scores["tone_score_without_decision"].diff()
)

# 5. Export both files
sentence_scores.to_csv("fomc_sentence_scores.csv", index=False)

statement_scores.to_csv(
    "fomc_statement_scores.csv",
    index=False
)

print("Sentence rows:", len(sentence_scores))
print("Statement rows:", len(statement_scores))
print("Unique sentence IDs:", sentence_scores["sentence_id"].nunique())
print("Unique statement IDs:", statement_scores["statement_id"].nunique())


Sentence rows: 2980
Statement rows: 227
Unique sentence IDs: 2980
Unique statement IDs: 227


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

from google.colab import files

files.download("fomc_sentence_scores.csv")
files.download("fomc_statement_scores.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

trying again